In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(context='notebook', style='darkgrid', font='sans-serif')

In [19]:
df = pd.read_csv('./titanic.csv')

In [20]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [21]:
df.tail()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.00,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.00,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


In [22]:
print(df.shape)

(891, 12)


In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [24]:
print(df.isnull().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [25]:
drop_cols = ['PassengerId', 'Cabin']

df = df.drop(columns=drop_cols)

In [26]:
df.columns

Index(['Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket',
       'Fare', 'Embarked'],
      dtype='str')

In [27]:
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

In [28]:
df["Age"] = df['Age'].fillna(df['Age'].median())

In [29]:
print(df.isnull().sum())

Survived    0
Pclass      0
Name        0
Sex         0
Age         0
SibSp       0
Parch       0
Ticket      0
Fare        0
Embarked    0
dtype: int64


In [30]:
df.head()

,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked
0,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C
2,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S
3,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,S
4,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,S


In [31]:
df["Sex"] = df.Sex.map({"male": 0, "female": 1})
df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)
df["FamilySize"] = df.SibSp + df.Parch + 1
df["IsAlone"] = (df.FamilySize == 1).astype(int)

In [32]:
df.columns

Index(['Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket',
       'Fare', 'Embarked_Q', 'Embarked_S', 'FamilySize', 'IsAlone'],
      dtype='str')

In [33]:
features = ["Pclass", "Sex", "Age", "Fare", "FamilySize", "IsAlone", "Embarked_Q", "Embarked_S"]
X = df[features]
y = df["Survived"]

In [34]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [35]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('sc', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

In [38]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

In [39]:
from sklearn.metrics import confusion_matrix, classification_report

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Confusion Matrix:
 [[94 16]
 [21 48]]

Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.85      0.84       110
           1       0.75      0.70      0.72        69

    accuracy                           0.79       179
   macro avg       0.78      0.78      0.78       179
weighted avg       0.79      0.79      0.79       179



In [40]:
y_probs = pipe.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
results = []


for th in thresholds:
    preds = (y_probs >= th).astype(int)
    
    results.append({
        "Chegara": th,
        "Accuracy": round(accuracy_score(y_test, preds), 4),
        "Precision": round(precision_score(y_test, preds, zero_division=0), 4),
        "Recall": round(recall_score(y_test, preds, zero_division=0), 4),
        "F1-score": round(f1_score(y_test, preds, zero_division=0), 4)
    })

In [43]:
res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))

# 2 xil variantda threshold tanlashimiz mumkin

# 1-variant: modelning umumiy sifati bizga muhim bolsa 0.6 chegara tanlanadi.
# Sababi: modelning accuracysi 82.12% chiqyabdi, precision va recall ortasidagi bog'liqlik (f1_score) ham maksimal

# 2-variant: hayotni saqlab qolish nuqtai nazaridan. bunda chegarani 0.3 tanlasak boladi
# Sababi: biz uchun recall omon qolganlarni iloji boricha koproq topish muhim. chegara 0.3 bolsa model omon qolganlarning 81.16% ni aniqlay oladi

 Chegara  Accuracy  Precision  Recall  F1-score
     0.1    0.5251     0.4444  0.9275    0.6009
     0.2    0.7207     0.5979  0.8406    0.6988
     0.3    0.7709     0.6667  0.8116    0.7320
     0.4    0.7821     0.7027  0.7536    0.7273
     0.5    0.7933     0.7500  0.6957    0.7218
     0.6    0.8212     0.8627  0.6377    0.7333
     0.7    0.7933     0.8810  0.5362    0.6667
     0.8    0.7318     0.9200  0.3333    0.4894
     0.9    0.6592     0.9000  0.1304    0.2278


In [44]:
df['IsChild'] = (df['Age'] < 16).astype(int)
df['FarePerPerson'] = df['Fare'] / df['FamilySize']

In [45]:
base_feats = ["Pclass", "Sex", "Age", "Fare", "FamilySize", "IsAlone", "Embarked_Q", "Embarked_S"]
y = df["Survived"]

In [46]:
from sklearn.model_selection import cross_val_score
def get_cv_score(features):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=42))
    ])
    scores = cross_val_score(pipe, df[features], y, cv=5, scoring='accuracy')
    return scores.mean(), scores.std()

base_score, base_std = get_cv_score(base_feats)
child_score, child_std = get_cv_score(base_feats + ['IsChild'])
fare_score, fare_std = get_cv_score(base_feats + ['FarePerPerson'])
both_score, both_std = get_cv_score(base_feats + ['IsChild', 'FarePerPerson'])

In [ ]:
print(f"8 ta feature bilan -> {round(base_score, 4)},   Base STD -> {round(base_std, 4)}")
print(f"faqat Child qo'shilganda -> {round(child_score, 4)},   Child STD -> {round(child_std, 4)}")
print(f"faqat Fare qoshilganda -> {round(fare_score, 4)},   Fare STD -> {round(fare_std, 4)}")
print(f"Ikkalasi ham qoshilganda-> {round(both_score, 4)},   Both STD -> {round(both_std, 4)}")

# bundan hulosa qiladigan bolsak bizga IsChild ustuni koproq yordam beryabdi va accuracyni oshiryabdi 

8 ta feature bilan -> 0.7924,   Base STD -> 0.0167
faqat Child qo'shilganda -> 0.8036,   Child STD -> 0.021
faqat Fare qoshilganda -> 0.7924,   Fare STD -> 0.0151
Ikkalasi ham qoshilganda-> 0.8025,   Both STD -> 0.0171
